In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Find the project's .env file
env_path = find_dotenv(usecwd=True)

# Load environment variables
load_dotenv(env_path)

# Read configuration
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_POLICY_DB = os.getenv("MYSQL_POLICY_DB")
MYSQL_BILLING_DB = os.getenv("MYSQL_BILLING_DB")

# Safe verification — password is NOT printed
print("Host:", MYSQL_HOST)
print("Port:", MYSQL_PORT)
print("User:", MYSQL_USER)
print("Policy DB:", MYSQL_POLICY_DB)
print("Billing DB:", MYSQL_BILLING_DB)
print("Password loaded:", bool(MYSQL_PASSWORD))

Host: localhost
Port: 3306
User: li_dev
Policy DB: li_policy_admin
Billing DB: li_billing
Password loaded: True


In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Build secure SQLAlchemy connection URL
policy_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=int(MYSQL_PORT),
    database=MYSQL_POLICY_DB
)

# Create SQLAlchemy engine
policy_engine = create_engine(policy_url)

# Test the connection
with policy_engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                VERSION() AS mysql_version,
                DATABASE() AS current_database,
                CURRENT_USER() AS authenticated_user
        """)
    )

    row = result.fetchone()

    print("MySQL Version:", row.mysql_version)
    print("Current Database:", row.current_database)
    print("Authenticated User:", row.authenticated_user)

MySQL Version: 9.2.0
Current Database: li_policy_admin
Authenticated User: li_dev@localhost


In [4]:
with policy_engine.connect() as connection:
    result = connection.execute(text("SHOW DATABASES"))

    databases = [row[0] for row in result]

    print("Visible databases:")
    for db in databases:
        print("-", db)

Visible databases:
- information_schema
- li_billing
- li_policy_admin
- performance_schema


In [5]:
import pandas as pd

# Build connection URL for billing database
billing_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=int(MYSQL_PORT),
    database=MYSQL_BILLING_DB
)

# Create billing engine
billing_engine = create_engine(billing_url)

# Query MySQL and return the result directly as a pandas DataFrame
billing_check = pd.read_sql_query(
    text("""
        SELECT
            VERSION() AS mysql_version,
            DATABASE() AS current_database,
            CURRENT_USER() AS authenticated_user
    """),
    billing_engine
)

billing_check

,mysql_version,current_database,authenticated_user
0,9.2.0,li_billing,li_dev@localhost
